In [0]:
/*
Static lookup tables to represent fiscal quarters.
*/
with dates as (
  select cast(m as date) as m, fq, fy, q
  , last_day(m) as last_day_of_month, year(m) as cy, month(m) as cm
  from (
    values 
      /*('2025-02-01', 'FY26-Q1', 2026, 1),
      ('2025-03-01', 'FY26-Q1', 2026, 1),
      ('2025-04-01', 'FY26-Q1', 2026, 1),
      ('2025-05-01', 'FY26-Q2', 2026, 2),
      ('2025-06-01', 'FY26-Q2', 2026, 2),
      ('2025-07-01', 'FY26-Q2', 2026, 2),
      ('2025-08-01', 'FY26-Q3', 2026, 3),
      ('2025-09-01', 'FY26-Q3', 2026, 3),
      ('2025-10-01', 'FY26-Q3', 2026, 3),*/
      ('2025-11-01', 'FY26-Q4', 2026, 4),
      ('2025-12-01', 'FY26-Q4', 2026, 4),
      ('2026-01-01', 'FY26-Q4', 2026, 4),
      ('2026-02-01', 'FY27-Q1', 2027, 1),
      ('2026-03-01', 'FY27-Q1', 2027, 1),
      ('2026-04-01', 'FY27-Q1', 2027, 1),
      ('2026-05-01', 'FY27-Q2', 2027, 2),
      ('2026-06-01', 'FY27-Q2', 2027, 2),
      ('2026-07-01', 'FY27-Q2', 2027, 2),
      ('2026-08-01', 'FY27-Q3', 2027, 3),
      ('2026-09-01', 'FY27-Q3', 2027, 3),
      ('2026-10-01', 'FY27-Q3', 2027, 3),
      ('2026-11-01', 'FY27-Q4', 2027, 4),
      ('2026-12-01', 'FY27-Q4', 2027, 4),
      ('2027-01-01', 'FY27-Q4', 2027, 4) 
  ) as dates(m, fq, fy, q)
),

-- Resolve :ae_email to a list of AE emails.
-- If :ae_email is an AE, returns just that email. If a manager, returns all AEs reporting to them.
ae_list as (
  select 
    user_id,
    Email as ae_email, 
    user_name, 
    IsAE, 
    case 
      when level = 7 then 'AE' 
      when level = 6 then 'BU+3 Lead' 
      when level = 5 then 'BU+2 Lead' 
      when level = 4 then 'BU+1 Lead' 
    end as sales_level, 
    crominus2name as bu_plus_1_lead, 
    crominus3name as bu_plus_2_lead, 
    crominus4name as bu_plus_3_lead,
    Business_Unit, Region_Level_1, Region_Level_2, Region_Level_3
  from main.gtm_silver.individual_hierarchy_salesforce
  where snapshot_date = (select max(snapshot_date) from main.gtm_silver.individual_hierarchy_salesforce)
  --and IsAE = true
  and IsActive = true
  and Business_Unit = :business_unit
  and Region_Level_1 = :region_level_1
  and Region_Level_2 = :region_level_2
  and concatenated_emails like '%' || :ae_email || '%'
),

account_region as (
  SELECT  
    m.account_id,
    m.territory_name,
    m.parent_name3 AS business_unit,
    m.parent_name2 AS subregion_level_1,
    m.parent_name1 AS subregion_level_2,
    m.parent_name0 AS subregion_level_3
  FROM main.gtm_silver.account_territory_map m --this preserves the original SFDC hierarchy names, useful to check HOLD accounts when at are assigned at intermin to another region level 3.
  WHERE m.parent_name3 = :business_unit
  AND m.parent_name2 = :region_level_1
  AND m.parent_name1 = :region_level_2
),

financial_quarters as (
  select fq, fy, q
    , max(last_day_of_month) as fiscal_quarter_end_date
    , min(m) as fiscal_quarter_start_date
    , case when getdate() > fiscal_quarter_end_date then q else null end as last_closed_q
    , case when getdate() > fiscal_quarter_end_date then true else false end as is_quarter_closed
    , sum(day(last_day_of_month)) as days_in_quarter
    , case when getdate() >= fiscal_quarter_start_date and current_date() <= fiscal_quarter_end_date then true else false end as is_current_fiscal_quarter
    , case when current_date() >= make_date(fy - 1, 2, 1) and current_date() <= make_date(fy, 1, 31) then 1 else 0 end as is_current_fiscal_year
    , (select max(usage_date) from main.gtm_gold.individual_consumption_daily)  as latest_usage_date
    , greatest(0, least(days_in_quarter, datediff(fiscal_quarter_end_date, latest_usage_date))) as days_left_in_quarter
    , case when is_current_fiscal_quarter then q else 0 end as current_quarter_number
  from dates
  group by fq, fy, q
),

targets as (
  SELECT ae.ae_email, t.Region_Level_1, t.Region_Level_2, t.Region_Level_3, t.user_id, t.dollars as fin_target, t.fiscal_year, concat("FY'", right(t.fiscal_year, 2), ' Q', t.fiscal_quarter) as fiscal_quarter, cast(t.fiscal_quarter as int) as fiscal_quarter_number
  FROM gtm_silver.targets_individual t
  INNER JOIN ae_list ae ON t.Email = ae.ae_email
  where t.Business_Unit = :business_unit
  and t.Region_Level_1 = :region_level_1
  and t.Region_Level_2 = :region_level_2
  and t.type_target = 'dbu'
  and t.snapshot_date = (select max(snapshot_date) from main.gtm_silver.targets_individual)
),

account_targets as (
  SELECT
    t.account_id, t.account_name, ae.ae_email
    , ih.Region_Level_1, ih.Region_Level_2, ih.Region_Level_3, ih.user_id
    , t.dbu_dollar_target as fin_target, t.fiscal_year, t.fiscal_quarter, cast(right(t.fiscal_quarter, 1) as int) as fiscal_quarter_number
  FROM main.gtm_silver.targets_account AS t
  INNER JOIN main.gtm_silver.account_dim ad
    ON t.account_id = ad.account_id
    AND ad.snapshot_date = (select max(snapshot_date) from main.gtm_silver.account_dim)
  INNER JOIN main.gtm_silver.individual_hierarchy_salesforce ih
    ON ad.account_executive_user_id = ih.user_id
    AND ih.snapshot_date = (select max(snapshot_date) from main.gtm_silver.individual_hierarchy_salesforce)
  INNER JOIN ae_list ae ON ih.user_id = ae.user_id 
  WHERE ih.Business_Unit = :business_unit
    AND ih.Region_Level_1 = :region_level_1
    AND ih.Region_Level_2 = :region_level_2
),

sales_forecast as (
  select ae.ae_email, f.forecast_owner_id as user_id, f.fiscal_quarter_end_date
  , coalesce(f.submitted_my_call, 0) as submitted_my_call
  , coalesce(f_prev.submitted_my_call, 0) as prev_submitted_my_call
  , coalesce(f.submitted_my_call_w_closed_month_actuals, 0) as submitted_my_call_w_closed_month_actuals
  , coalesce(f.submitted_direct_field_consumption_forecast, 0) as submitted_direct_field_consumption_forecast
  , coalesce(f.current_ds_forecast_all_accounts, 0) as current_ds_forecast
  , coalesce(f.submitted_weighted_projection, 0) as submitted_weighted_projection
  , ae.bu_plus_2_lead, ae.bu_plus_3_lead, ae.sales_level, ae.user_name
  from gtm_silver.forecast_consumption_mcp_individual as f
  inner join ae_list ae on f.Email = ae.ae_email
  left join gtm_silver.forecast_consumption_mcp_individual as f_prev
    on f_prev.Email = f.Email
    and f_prev.fiscal_quarter_end_date = f.fiscal_quarter_end_date
    and f_prev.snapshot_date = (select max(snapshot_date) from main.gtm_silver.forecast_consumption_mcp_individual where snapshot_date < (select max(snapshot_date) from main.gtm_silver.forecast_consumption_mcp_individual))
  where f.Business_Unit = :business_unit
  and f.Region_Level_1 = :region_level_1
  and f.Region_Level_2 = :region_level_2
  and f.snapshot_date = (select max(snapshot_date) from main.gtm_silver.forecast_consumption_mcp_individual)
),

account_forecast as (
select 
  f.account_id, a.account_name, i.Email as ae_email, f.account_executive_user_id as user_id, 
  f.forecast_fiscal_quarter_end_date as fiscal_quarter_end_date
  , ae.bu_plus_2_lead, ae.bu_plus_3_lead, ae.sales_level, ae.user_name
  , coalesce(sum(f.submitted_ae_forecast), 0) as submitted_my_call
  , coalesce(sum(f_prev.submitted_ae_forecast), 0) as prev_submitted_my_call
  , coalesce(sum(f.submitted_ae_forecast), 0) as submitted_direct_field_consumption_forecast
  , coalesce(sum(f.submitted_ae_forecast_w_closed_month_actuals), 0) as submitted_my_call_w_closed_month_actuals
  , coalesce(sum(f.current_ds_forecast), 0) as current_ds_forecast
  , coalesce(sum(f.submitted_weighted_projection), 0) as submitted_weighted_projection
  from gtm_silver.forecast_consumption_mcp_account as f
  inner join ae_list ae on f.account_executive_user_id = ae.user_id
  left join main.gtm_silver.account_dim as a
    on f.account_id = a.account_id
    and a.snapshot_date = (select max(snapshot_date) from main.gtm_silver.account_dim)
  inner join main.gtm_silver.individual_hierarchy_salesforce as i
    on f.account_executive_user_id = i.user_id
    and i.snapshot_date = (select max(snapshot_date) from main.gtm_silver.individual_hierarchy_salesforce)  
  left join gtm_silver.forecast_consumption_mcp_account as f_prev
    on f_prev.account_id = f.account_id
    and f_prev.account_executive_user_id = f.account_executive_user_id
    and f_prev.forecast_fiscal_quarter_end_date = f.forecast_fiscal_quarter_end_date
    and f_prev.forecast_month_start_date = f.forecast_month_start_date
    and f_prev.snapshot_date = (select max(snapshot_date) from main.gtm_silver.forecast_consumption_mcp_account where snapshot_date < (select max(snapshot_date) from main.gtm_silver.forecast_consumption_mcp_account))
  where i.Business_Unit = :business_unit
  and i.Region_Level_1 = :region_level_1
  and i.Region_Level_2 = :region_level_2
  and f.snapshot_date = (select max(snapshot_date) from main.gtm_silver.forecast_consumption_mcp_account)
  group by all
),

actuals as (
  select :ae_email as selected_user, ae.user_id, ae.ae_email, c.fiscal_quarter_start_date
  , ae.bu_plus_2_lead, ae.bu_plus_3_lead, ae.sales_level, ae.user_name
  , ae.business_unit AS Business_Unit, ae.region_level_1 AS Region_Level_1, ae.region_level_2 AS Region_Level_2, ae.region_level_3 AS Region_Level_3
  , sum(c.dbu_dollars_qtd) as dbu_actuals
  , sum(c.dbu_dollars_t7d_avg) as dbu_dollars_t7d_avg, sum(c.dbu_dollars_t28d_avg) as dbu_dollars_t28d_avg
  , sum(c.dbu_dollars_t7d_avg_prev) as dbu_dollars_t7d_avg_prev, sum(c.dbu_dollars_t28d_avg_prev) as dbu_dollars_t28d_avg_prev
  from ae_list ae
  inner join main.gtm_gold.materialized__view_account_obt as c
    on c.concatenated_emails like '%' || ae.ae_email || '%' -- I need the Sales hierarchy, not just the AE
  left outer join main.gtm_silver.account_dim as b
    on c.account_id = b.account_id
  --left outer join account_region as ar
  --on ar.account_id = c.account_id
  where b.business_unit = :business_unit
  and b.region_level_1 = :region_level_1
  and b.region_level_2 = :region_level_2
  group by all
),

account_actuals as (
  select :ae_email as selected_user, ae.user_id, ae.ae_email
  , ae.bu_plus_2_lead, ae.bu_plus_3_lead, ae.sales_level, ae.user_name
  , ar.business_unit AS Business_Unit, ar.subregion_level_1 AS Region_Level_1, ar.subregion_level_2 AS Region_Level_2, ar.subregion_level_3 AS Region_Level_3
  , c.account_id, c.account_name, c.fiscal_quarter_start_date
  , coalesce(sum(c.dbu_dollars_qtd), 0) as dbu_actuals
  , coalesce(sum(c.dbu_dollars_t7d_avg), 0) as dbu_dollars_t7d_avg
  , coalesce(sum(c.dbu_dollars_t28d_avg), 0) as dbu_dollars_t28d_avg
  , coalesce(sum(c.dbu_dollars_t7d_avg_prev), 0) as dbu_dollars_t7d_avg_prev
  , coalesce(sum(c.dbu_dollars_t28d_avg_prev), 0) as dbu_dollars_t28d_avg_prev
  from ae_list ae
  inner join main.gtm_gold.materialized__view_account_obt as c
    on c.account_executive_user_id = ae.user_id
    --on c.concatenated_emails like '%' || ae.ae_email || '%' --
  left outer join main.gtm_silver.account_dim as b
    on c.account_id = b.account_id
  left outer join account_region as ar
    on ar.account_id = c.account_id
  where b.business_unit = :business_unit
  and b.region_level_1 = :region_level_1
  and b.region_level_2 = :region_level_2
  group by all
),

target_forecast_actuals as (
  select a.selected_user, a.user_id, a.ae_email, a.bu_plus_2_lead, a.bu_plus_3_lead, a.sales_level, a.user_name
    , a.Business_Unit, a.Region_Level_1, a.Region_Level_2, a.Region_Level_3
    , d.fy, d.q, d.fq, a.fiscal_quarter_start_date, a.dbu_actuals
    , a.dbu_dollars_t7d_avg, a.dbu_dollars_t28d_avg, a.dbu_dollars_t7d_avg_prev, a.dbu_dollars_t28d_avg_prev
    , t.fin_target, f.submitted_my_call, f.submitted_direct_field_consumption_forecast, f.current_ds_forecast, f.submitted_weighted_projection
    , coalesce(try_divide(f.submitted_my_call - f.prev_submitted_my_call, nullif(f.prev_submitted_my_call, 0)), 0) as forecast_change_pct
    , coalesce(a.dbu_actuals, 0) as dbu_actuals_coalesced
    , coalesce(case when d.is_quarter_closed then a.dbu_actuals else submitted_my_call end, 0) as dbu_actuals_or_forecast
    , first_value(a.dbu_dollars_t7d_avg) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_latest
    , coalesce(nullif(a.dbu_dollars_t7d_avg, 0), dbu_dollars_t7d_avg_latest) as dbu_dollars_t7d_adj --for future quarters, use the latest t7d available.
    , first_value(a.dbu_dollars_t28d_avg) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_latest
    , coalesce(nullif(a.dbu_dollars_t28d_avg, 0), dbu_dollars_t28d_avg_latest) as dbu_dollars_t28d_adj --for future quarters, use the latest t28d available.
    , first_value(a.dbu_dollars_t7d_avg_prev) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_prev_latest
    , coalesce(nullif(a.dbu_dollars_t7d_avg_prev, 0), dbu_dollars_t7d_avg_prev_latest) as dbu_dollars_t7d_prev_adj --for future quarters, use the latest t7d_prev available.
    , first_value(a.dbu_dollars_t28d_avg_prev) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_prev_latest
    , coalesce(nullif(a.dbu_dollars_t28d_avg_prev, 0), dbu_dollars_t28d_avg_prev_latest) as dbu_dollars_t28d_prev_adj --for future quarters, use the latest t28d_prev available.
    , case when d.is_quarter_closed then 0 else dbu_actuals_coalesced end dbu_actuals_current_quarter
    , case when d.is_quarter_closed then 0 else (dbu_dollars_t7d_adj * d.days_left_in_quarter) end as t7d_proj_left_in_quarter
    , case when d.is_quarter_closed then 0 else (dbu_dollars_t28d_adj * d.days_left_in_quarter) end as t28d_proj_left_in_quarter
    , coalesce(try_divide(f.submitted_my_call - dbu_actuals_coalesced, d.days_left_in_quarter), 0) as target_t7d  

  from actuals as a
  inner join financial_quarters d
  on d.fiscal_quarter_start_date = a.fiscal_quarter_start_date
  left outer join sales_forecast as f 
  on f.fiscal_quarter_end_date = d.fiscal_quarter_end_date and f.ae_email = a.ae_email
  left outer join targets as t 
  on t.fiscal_year = d.fy and t.fiscal_quarter_number = d.q and t.user_id = a.user_id
),

account_target_forecast_actuals as (
  select a.selected_user, a.user_id, a.bu_plus_2_lead, a.bu_plus_3_lead, a.sales_level, a.user_name
    , a.Business_Unit, a.Region_Level_1, a.Region_Level_2, a.Region_Level_3
    , a.ae_email, a.account_id, a.account_name, a.fiscal_quarter_start_date, d.fy, d.q, d.fq
    , coalesce(t.fin_target, 0) as fin_target
    , coalesce(f.submitted_my_call, 0) as submitted_my_call
    , coalesce(f.submitted_direct_field_consumption_forecast, 0) as submitted_direct_field_consumption_forecast
    , coalesce(f.current_ds_forecast, 0) as current_ds_forecast
    , coalesce(f.submitted_weighted_projection, 0) as submitted_weighted_projection
    , coalesce(try_divide(f.submitted_my_call - f.prev_submitted_my_call, nullif(f.prev_submitted_my_call, 0)), 0) as forecast_change_pct
    , coalesce(a.dbu_actuals, 0) as dbu_actuals
    , coalesce(a.dbu_dollars_t7d_avg, 0) as dbu_dollars_t7d_avg
    , coalesce(a.dbu_dollars_t28d_avg, 0) as dbu_dollars_t28d_avg
    , coalesce(a.dbu_dollars_t7d_avg_prev, 0) as dbu_dollars_t7d_avg_prev
    , coalesce(a.dbu_dollars_t28d_avg_prev, 0) as dbu_dollars_t28d_avg_prev    
    , coalesce(a.dbu_actuals, 0) as dbu_actuals_coalesced
    , coalesce(case when d.is_quarter_closed then a.dbu_actuals else f.submitted_my_call end, 0) as dbu_actuals_or_forecast
    , first_value(a.dbu_dollars_t7d_avg) over(partition by a.ae_email, a.account_name order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_latest
    , coalesce(nullif(a.dbu_dollars_t7d_avg, 0), dbu_dollars_t7d_avg_latest) as dbu_dollars_t7d_adj
    , first_value(a.dbu_dollars_t28d_avg) over(partition by a.ae_email, a.account_name order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_latest
    , coalesce(nullif(a.dbu_dollars_t28d_avg, 0), dbu_dollars_t28d_avg_latest) as dbu_dollars_t28d_adj
    , first_value(a.dbu_dollars_t7d_avg_prev) over(partition by a.ae_email, a.account_name order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_prev_latest
    , coalesce(nullif(a.dbu_dollars_t7d_avg_prev, 0), dbu_dollars_t7d_avg_prev_latest) as dbu_dollars_t7d_prev_adj
    , first_value(a.dbu_dollars_t28d_avg_prev) over(partition by a.ae_email, a.account_name order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_prev_latest
    , coalesce(nullif(a.dbu_dollars_t28d_avg_prev, 0), dbu_dollars_t28d_avg_prev_latest) as dbu_dollars_t28d_prev_adj
    , case when d.is_quarter_closed then 0 else dbu_actuals_coalesced end dbu_actuals_current_quarter
    , case when d.is_quarter_closed then 0 else (dbu_dollars_t7d_adj * d.days_left_in_quarter) end as t7d_proj_left_in_quarter
    , case when d.is_quarter_closed then 0 else (dbu_dollars_t28d_adj * d.days_left_in_quarter) end as t28d_proj_left_in_quarter
    , coalesce(try_divide(f.submitted_my_call - dbu_actuals_coalesced, d.days_left_in_quarter), 0) as target_t7d

  from account_actuals as a
  inner join financial_quarters d
    on d.fiscal_quarter_start_date = a.fiscal_quarter_start_date
  left outer join account_forecast as f
    on f.fiscal_quarter_end_date = d.fiscal_quarter_end_date and f.user_id = a.user_id and f.account_id = a.account_id
  left outer join account_targets as t
    on t.fiscal_year = d.fy and t.fiscal_quarter_number = d.q and t.user_id = a.user_id and t.account_id = a.account_id
),

usecases_filtered as (
  select ae.user_id, ae.ae_email, usecase_id, usecase_name, account_name, estimated_monthly_dollar_dbus, target_onboarding_date, target_live_date
    , dateadd(day, 14, date_trunc('month',target_onboarding_date)) as target_onboarding_date_15 
    , dateadd(day, 14, date_trunc('month',target_live_date)) as target_live_date_15
    , datediff(target_live_date, target_onboarding_date) as total_ramping_days 
    , date_format(dateadd(year, +1, dateadd(month, -1, target_onboarding_date)), "'FY'yy'-Q'Q") as target_onboarding_date_fq
    , date_format(dateadd(year, +1, dateadd(month, -1, target_live_date)), "'FY'yy'-Q'Q") as target_live_date_fq
    , concat('',usecase_name,'') as usecase_url
    , coalesce(num_of_blockers, 0) as num_of_blockers
    , case
        when days_in_stage <= 30 or days_in_stage is null then '0-30 days'
        when days_in_stage > 30 and days_in_stage <= 60 then '31-60 days'
        when days_in_stage > 60 and days_in_stage <= 120 then '61-120 days'
        when days_in_stage > 120 then '120+ days'
      end as days_in_stage_bucket
    , date_diff(DAY, current_date(), last_day(target_live_date)) as days_to_go_live
    , case when days_to_go_live < 0 then true else false end go_live_in_the_past
    , date_diff(DAY, current_date(), last_day(target_onboarding_date)) as days_to_onboarding

     --Check hygiene issues and risks
    , case
        when go_live_in_the_past then named_struct('category', 'Hygiene', 'msg', 'Go live date in the past')
        when implementation_status is null then named_struct('category', 'Hygiene', 'msg', 'Health status not defined')
        when days_to_onboarding < 0 and stage_number < 5 then named_struct('category', 'Hygiene', 'msg', 'Past Onboarding date / not U5') 
        when days_to_go_live < 30 and stage_number < 5 then named_struct('category', 'Warning', 'msg', 'Go live < 30 / Not U5')
        when days_to_go_live < 30 and implementation_status = 'Red' then named_struct('category', 'Warning', 'msg', 'Go live < 30 days / Red')
        when days_to_go_live < 30 and implementation_status = 'Yellow' then named_struct('category', 'Warning', 'msg', 'Go live < 30 days / Yellow')
        when days_to_go_live < 30 then named_struct('category', 'Warning', 'msg', 'Go live < 30')
        when days_to_onboarding between 0 and 30 and stage_number < 5 and implementation_status = 'Red' then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / Red')
        when days_to_onboarding between 0 and 30 and stage_number < 5 and implementation_status = 'Yellow' then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / Yellow')
        when days_to_onboarding between 0 and 30 and stage_number <= 3 then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / <=U3')
        when days_to_onboarding between 0 and 30 and stage_number = 4 then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / U4')
        when days_to_go_live < 60 and stage_number = 5 then named_struct('category', 'Opportunity', 'msg', 'Go live < 60 / U5')
        when days_to_onboarding between 0 and 60 and stage_number between 2 and 3 then named_struct('category', 'Opportunity', 'msg', 'Tech win to accelerate')
      end as uco_info
  , coalesce(implementation_status, 'Unknown') as impl_status
    
  from gtm_silver.use_case_detail
  inner join ae_list ae on use_case_detail.concatenated_emails like '%' || ae.ae_email || '%'
  where use_case_detail.Business_Unit = :business_unit
  and sales_subregion_level_1 = :region_level_1
  and sales_subregion_level_2 = :region_level_2
  and is_incremental = true --Excludes upgrades
  and stage_number <= 5 --Filter out 'Disqualified', 'Lost' and 'Live' UCOs.
  and coalesce(estimated_monthly_dollar_dbus, 0) > 0 -- Excludes zero-valued use cases.
  --and usecase_id = 'aAv8Y000000CsLuSAK' 
  /* test cases 
  aAv8Y000000CsLuSAK (Feb25->Sep25), aAvVp000000Uc6IKAS (May25->Sep25), aAv8Y000000lD0ySAE (Jun25->Dec25)
  aAvVp000000W9JVKA0 (Apr25->May25), aAvVp000000d2YEKAY (Feb25->Jul25), aAvVp000000WAqfKAG (Jun25->Jan26)
  */
),

incremental_projections as (
  select uco.user_id, uco.ae_email, uco.usecase_id, uco.account_name, d.fq, d.fy, d.q, d.m, d.cm, d.last_day_of_month
    , uco.target_onboarding_date, uco.target_onboarding_date_fq, uco.target_live_date, uco.target_live_date_fq, uco.target_onboarding_date_15, uco.target_live_date_15
    , uco.total_ramping_days, uco.estimated_monthly_dollar_dbus, uco.impl_status, uco.usecase_url, uco.num_of_blockers 
    , (select max(usage_date) from main.gtm_gold.individual_consumption_daily) as latest_usage_date
    , datediff(latest_usage_date, target_onboarding_date_15) as current_ramping_days 
    -- Calculate this month's baseline for each use case, .i.e. how much are they consuming today? This is used to calculate the actual incremental consumption at the next step.
    -- We assume that the onboarding date and live date occur on day 15 of the month.
    ,case when d.m between uco.target_onboarding_date and uco.target_live_date then 1 else 0 end as is_onboarding
  
    ,case         
      -- if UCO not onboarded yet (i.e. the onboarding date is in the future), then no dbus are generated for the current month.
      when target_onboarding_date_15 > latest_usage_date then 0
      -- if UCO is already live, then it should already realise the expected monthly $DBUs.
      when latest_usage_date > target_live_date_15 then estimated_monthly_dollar_dbus
      -- if UCO is currently onboarding (i.e. the onboarding date is in the past), this is the estimated dbus for the full current month. 
      else round(estimated_monthly_dollar_dbus * try_divide(datediff(latest_usage_date, target_onboarding_date_15), total_ramping_days)) 
    end as current_dbu_baseline 

    ,case
        when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
        when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus --After the go-live the $dbus remain flat
        else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) --Between onboarding and go-live the dbus ramp-up linearly
      end as ramping_dbus

    --remove the realised dbus from the ramp, when the use case is ramping up during the onboarding phase, past months' revenue has already been realised.
    , case 
      when d.last_day_of_month < latest_usage_date then 0 --Past month: if a use case started onboarding in the past, and the month is closed then we are removing the consumption from the pipeline to avoid double counting, because we assume it has already been realised (actual dbus).
      when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
      when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus - current_dbu_baseline --After the go-live
      else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) - current_dbu_baseline --Between onboarding and go-live 
    end as ramping_dbus_from_baseline 

     -- calculates the actual incremental value substracting last month's $dbus from this month's $dbus.
    , case 
        when latest_usage_date > m then ramping_dbus - ramping_dbus_from_baseline
        else 0 --in the future
    end as dbus_generated

  from usecases_filtered as uco
  inner join dates as d --cross join with date table
),

quarterly_projection_by_use_case as (
  select              
    i.user_id, i.usecase_id, i.fy, i.fq, i.q
    , sum(i.ramping_dbus) as quarterly_ramping_dbus    
    , sum(i.dbus_generated) as quarterly_dbus_generated
    , lag(max(i.ramping_dbus)) over (partition by i.user_id, i.usecase_id order by i.fq asc) as last_day_of_prev_quarter_dbus
    from incremental_projections as i    
    group by all
),

monthly_projection as (
  select
    ip.user_id, ip.ae_email, ip.usecase_id, ip.account_name, ip.fy, ip.fq, ip.q, ip.m, ip.cm, ip.ramping_dbus, ip.current_dbu_baseline, ip.is_onboarding
    , f.last_closed_q, f.days_left_in_quarter, f.is_quarter_closed, f.is_current_fiscal_quarter
    , qp.last_day_of_prev_quarter_dbus    
    , greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline) as quarter_dbu_baseline

    -- Incremental quarterly $dbus. 
    -- If the quarter has started then we use the current baseline to identify addtional incremental dbus until the end of the quarter
    -- if the quarter has not started yet the baseline is the last day of the previous quarter.    
    ,coalesce(
        case 
            when ip.ramping_dbus - greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline) < 0 then 0 -- All past months do not contribute to incremental dbus. 
            else ip.ramping_dbus - greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline)
        end, 0) as quarterly_incremental_dbus

    , coalesce(
        case when ip.impl_status = 'Green' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end, 0) as dbus_in_pipeline_green 
    , coalesce(
        case when ip.impl_status = 'Yellow' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end, 0) as dbus_in_pipeline_yellow 
    , coalesce(
        case when ip.impl_status = 'Red' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end, 0) as dbus_in_pipeline_red
    , coalesce(
        case when ip.impl_status = 'Unknown' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end, 0) as dbus_in_pipeline_unknown
  
  from incremental_projections as ip
  inner join quarterly_projection_by_use_case as qp
  on ip.usecase_id = qp.usecase_id
  and ip.fq = qp.fq
  and ip.user_id = qp.user_id
  inner join financial_quarters as f
  on ip.fq = f.fq
),

account_quarterly_projection as (
  select p.user_id, p.account_name, p.ae_email, p.fy, p.fq, p.q, p.last_closed_q, p.days_left_in_quarter, p.is_quarter_closed, p.is_current_fiscal_quarter
    , coalesce(sum(p.quarterly_incremental_dbus), 0) as quarterly_incremental_dbus
    , coalesce(sum(p.dbus_in_pipeline_green), 0) as dbus_in_pipeline_green 
    , coalesce(sum(p.dbus_in_pipeline_yellow), 0) as dbus_in_pipeline_yellow 
    , coalesce(sum(p.dbus_in_pipeline_red), 0) as dbus_in_pipeline_red
    , coalesce(sum(p.dbus_in_pipeline_unknown), 0) as dbus_in_pipeline_unknown
    , coalesce(sum(last_day_of_prev_quarter_dbus), 0) as last_day_of_prev_quarter_dbus

    from monthly_projection as p
    group by all
),

quarterly_projection as (
  select p.user_id, p.ae_email, p.fy, p.fq, p.q, p.last_closed_q, p.days_left_in_quarter, p.is_quarter_closed, p.is_current_fiscal_quarter
    , coalesce(sum(p.quarterly_incremental_dbus), 0) as quarterly_incremental_dbus
    , coalesce(sum(p.dbus_in_pipeline_green), 0) as dbus_in_pipeline_green 
    , coalesce(sum(p.dbus_in_pipeline_yellow), 0) as dbus_in_pipeline_yellow 
    , coalesce(sum(p.dbus_in_pipeline_red), 0) as dbus_in_pipeline_red
    , coalesce(sum(p.dbus_in_pipeline_unknown), 0) as dbus_in_pipeline_unknown
    , coalesce(sum(last_day_of_prev_quarter_dbus), 0) as last_day_of_prev_quarter_dbus

    from account_quarterly_projection as p
    group by all
),

account_union_individuals as (
  -- Individual level
  select
    case when f.selected_user = f.ae_email then 'Individual Forecast' else 'Org Forecast' end as view_name
    --'Individual Forecast' as view_name
    , cast(NULL as string) as account_id
    , 'All' as account_name
    , f.selected_user, f.user_id, f.ae_email, f.bu_plus_2_lead, f.bu_plus_3_lead, f.sales_level, f.user_name
    , f.Business_Unit, f.Region_Level_1, f.Region_Level_2, f.Region_Level_3
    , f.fy, f.q, f.fq, f.fiscal_quarter_start_date    
    , f.dbu_actuals
    , f.dbu_dollars_t7d_avg, f.dbu_dollars_t28d_avg, f.dbu_dollars_t7d_avg_prev, f.dbu_dollars_t28d_avg_prev
    , f.fin_target, f.submitted_my_call, f.submitted_direct_field_consumption_forecast, f.current_ds_forecast, f.submitted_weighted_projection
    , f.forecast_change_pct
    , f.dbu_actuals_coalesced, f.dbu_actuals_or_forecast
    , f.dbu_dollars_t7d_avg_latest, f.dbu_dollars_t7d_adj
    , f.dbu_dollars_t28d_avg_latest, f.dbu_dollars_t28d_adj
    , f.dbu_dollars_t7d_avg_prev_latest, f.dbu_dollars_t7d_prev_adj
    , f.dbu_dollars_t28d_avg_prev_latest, f.dbu_dollars_t28d_prev_adj
    , f.dbu_actuals_current_quarter, f.t7d_proj_left_in_quarter, f.t28d_proj_left_in_quarter, f.target_t7d
    , coalesce(p.last_closed_q, 0) as last_closed_q
    , coalesce(p.days_left_in_quarter, 0) as days_left_in_quarter
    , coalesce(p.is_quarter_closed, false) as is_quarter_closed
    , coalesce(p.is_current_fiscal_quarter, false) as is_current_fiscal_quarter
    , coalesce(p.quarterly_incremental_dbus, 0) as quarterly_incremental_dbus
    , coalesce(p.dbus_in_pipeline_green, 0) as dbus_in_pipeline_green
    , coalesce(p.dbus_in_pipeline_yellow, 0) as dbus_in_pipeline_yellow
    , coalesce(p.dbus_in_pipeline_red, 0) as dbus_in_pipeline_red
    , coalesce(p.dbus_in_pipeline_unknown, 0) as dbus_in_pipeline_unknown
    , coalesce(p.last_day_of_prev_quarter_dbus, 0) as last_day_of_prev_quarter_dbus
  from target_forecast_actuals as f
  left outer join quarterly_projection as p
    on p.fy = f.fy and p.q = f.q and p.ae_email = f.ae_email

  UNION ALL

  -- Account level
  select
    'Account Forecast' as view_name
    , f.account_id, f.account_name
    , f.selected_user, f.user_id, f.ae_email, f.bu_plus_2_lead, f.bu_plus_3_lead, f.sales_level, f.user_name
    , f.Business_Unit, f.Region_Level_1, f.Region_Level_2, f.Region_Level_3
    , f.fy, f.q, f.fq, f.fiscal_quarter_start_date    
    , f.dbu_actuals
    , f.dbu_dollars_t7d_avg, f.dbu_dollars_t28d_avg, f.dbu_dollars_t7d_avg_prev, f.dbu_dollars_t28d_avg_prev
    , f.fin_target, f.submitted_my_call, f.submitted_direct_field_consumption_forecast, f.current_ds_forecast, f.submitted_weighted_projection
    , f.forecast_change_pct
    , f.dbu_actuals_coalesced, f.dbu_actuals_or_forecast
    , f.dbu_dollars_t7d_avg_latest, f.dbu_dollars_t7d_adj
    , f.dbu_dollars_t28d_avg_latest, f.dbu_dollars_t28d_adj
    , f.dbu_dollars_t7d_avg_prev_latest, f.dbu_dollars_t7d_prev_adj
    , f.dbu_dollars_t28d_avg_prev_latest, f.dbu_dollars_t28d_prev_adj
    , f.dbu_actuals_current_quarter, f.t7d_proj_left_in_quarter, f.t28d_proj_left_in_quarter, f.target_t7d
    , coalesce(p.last_closed_q, 0) as last_closed_q
    , coalesce(p.days_left_in_quarter, 0) as days_left_in_quarter
    , coalesce(p.is_quarter_closed, false) as is_quarter_closed
    , coalesce(p.is_current_fiscal_quarter, false) as is_current_fiscal_quarter
    , coalesce(p.quarterly_incremental_dbus, 0) as quarterly_incremental_dbus
    , coalesce(p.dbus_in_pipeline_green, 0) as dbus_in_pipeline_green
    , coalesce(p.dbus_in_pipeline_yellow, 0) as dbus_in_pipeline_yellow
    , coalesce(p.dbus_in_pipeline_red, 0) as dbus_in_pipeline_red
    , coalesce(p.dbus_in_pipeline_unknown, 0) as dbus_in_pipeline_unknown
    , coalesce(p.last_day_of_prev_quarter_dbus, 0) as last_day_of_prev_quarter_dbus
  from account_target_forecast_actuals as f
  left outer join account_quarterly_projection as p
    on p.fy = f.fy and p.q = f.q and p.user_id = f.user_id and p.account_name = f.account_name
),

quarterly_summary as (
  SELECT 
  u.view_name, u.ae_email, u.account_id, u.account_name, u.sales_level, u.user_name, u.bu_plus_2_lead, u.bu_plus_3_lead, u.Business_Unit, u.Region_Level_1, u.Region_Level_2, u.Region_Level_3, u.fy, u.fq, u.days_left_in_quarter, u.is_quarter_closed , u.is_current_fiscal_quarter
  , u.fin_target, u.submitted_my_call 
  , u.submitted_direct_field_consumption_forecast, u.current_ds_forecast, u.submitted_weighted_projection
  , u.forecast_change_pct
  , u.dbu_actuals_coalesced as dbu_actuals, u.dbu_actuals_or_forecast, u.dbu_actuals_current_quarter
  , u.dbu_dollars_t7d_adj, u.dbu_dollars_t7d_prev_adj, u.t7d_proj_left_in_quarter, u.target_t7d
  , u.dbu_dollars_t28d_adj, u.dbu_dollars_t28d_prev_adj, u.t28d_proj_left_in_quarter

  , coalesce(lag(u.dbu_actuals_or_forecast) over (partition by u.user_id, u.account_id order by u.fq asc), 0) as dbu_actuals_or_forecast_prev_quarter
  , coalesce(try_divide(u.fin_target - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), 0) as qoq_target_growth

  --incremental pipeline
  , u.quarterly_incremental_dbus 
  , u.dbus_in_pipeline_green, u.dbus_in_pipeline_yellow, u.dbus_in_pipeline_red, u.dbus_in_pipeline_unknown
  , u.dbus_in_pipeline_green * :green_confidence_pct as dbus_in_pipeline_green_in_plan
  , u.dbus_in_pipeline_yellow * :yellow_confidence_pct as dbus_in_pipeline_yellow_in_plan
  , u.dbus_in_pipeline_red * :red_confidence_pct as dbus_in_pipeline_red_in_plan
  , u.dbus_in_pipeline_unknown * :unknown_confidence_pct as dbus_in_pipeline_unknown_in_plan

  --set the baseline: 
  --for the current quarter, use the T7D because more precise. 
  --for future quarters use 
  , case when u.is_quarter_closed then 0 when u.is_current_fiscal_quarter then u.t7d_proj_left_in_quarter else dbu_actuals_or_forecast_prev_quarter  end as baseline
  , case when u.is_quarter_closed then "Baseline" when u.is_current_fiscal_quarter then "Baseline (T7D Projection + OG)" else "Baseline (previous quarter's forecast) + OG" end as baseline_label -- used as a label in the dashboard

  , case when u.is_quarter_closed then 0 else u.dbu_actuals_coalesced + u.t7d_proj_left_in_quarter end as t7d_flat_projection 
  , case when u.is_quarter_closed then 0 else u.dbu_actuals_coalesced + u.t28d_proj_left_in_quarter end as t28d_flat_projection

  --best case
  , baseline * :best_case_qoq_organic_growth as best_case_proj_with_og_left_in_quarter
  , dbus_in_pipeline_green_in_plan + dbus_in_pipeline_yellow_in_plan + dbus_in_pipeline_red_in_plan + dbus_in_pipeline_unknown_in_plan as best_case_pipe_left_in_quarter
  , u.dbu_actuals_coalesced + best_case_proj_with_og_left_in_quarter + best_case_pipe_left_in_quarter + :best_case_adjustments as best_case_projection

  --worst case
  , baseline * :worst_case_qoq_organic_growth as worst_case_proj_with_og_left_in_quarter
  , dbus_in_pipeline_green_in_plan as worst_case_pipe_left_in_quarter
  , u.dbu_actuals_coalesced + worst_case_proj_with_og_left_in_quarter + worst_case_pipe_left_in_quarter + :worst_case_adjustments as worst_case_projection
  
  --forecast gaps
  , u.submitted_my_call - u.fin_target as gap_my_call
  , best_case_projection - u.fin_target as gap_best_case
  , worst_case_projection - u.fin_target as gap_worst_case
  , t7d_flat_projection - u.fin_target as gap_t7d_flat_projection
  , t28d_flat_projection - u.fin_target as gap_t28d_flat_projection
  , u.submitted_weighted_projection - u.fin_target as gap_weighted_projection
  , u.current_ds_forecast - u.fin_target as gap_ds_forecast
  , u.submitted_direct_field_consumption_forecast - u.fin_target as gap_directs_forecast
  , u.submitted_my_call - u.submitted_direct_field_consumption_forecast as gap_my_call_vs_directs

   --QoQ delta: used to calculate the QoQ % groeth
  , u.submitted_my_call - dbu_actuals_or_forecast_prev_quarter as qoq_delta_target 
  , best_case_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_best_case
  , worst_case_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_worst_case
  , t7d_flat_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_t7d_flat_projection
  , t28d_flat_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_t28d_flat_projection
  , u.submitted_weighted_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_weighted_projection
  , u.current_ds_forecast - dbu_actuals_or_forecast_prev_quarter as qoq_delta_ds_forecast
  , u.submitted_direct_field_consumption_forecast - dbu_actuals_or_forecast_prev_quarter as qoq_delta_directs_forecast

  , coalesce(try_divide(u.submitted_my_call, u.fin_target), 0) as my_call_att
  , coalesce(try_divide(qoq_delta_target, dbu_actuals_or_forecast_prev_quarter), 0) as my_call_qoq_perc
  , coalesce(try_divide(u.submitted_direct_field_consumption_forecast, u.fin_target), 0) as directs_att
  , coalesce(try_divide(qoq_delta_directs_forecast, dbu_actuals_or_forecast_prev_quarter), 0) as directs_qoq_perc
  , coalesce(u.dbu_dollars_t7d_adj - u.dbu_dollars_t7d_prev_adj, 0) as t7d_change_dollar_dbu
  , coalesce(u.dbu_dollars_t28d_adj - u.dbu_dollars_t28d_prev_adj, 0) as t28d_change_dollar_dbu
  , coalesce(try_divide(t7d_change_dollar_dbu, u.dbu_dollars_t7d_prev_adj), 0) as t7d_change_perc
  , coalesce(try_divide(t28d_change_dollar_dbu, u.dbu_dollars_t28d_prev_adj), 0) as t28d_change_perc

  --labels
  , format_number(try_divide(u.submitted_my_call, u.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_target, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_my_call, '$,###.#') as my_call_text
  , format_number(try_divide(u.submitted_direct_field_consumption_forecast, u.fin_target), '#.#%') ||
    " | " || format_number(try_divide(qoq_delta_directs_forecast, dbu_actuals_or_forecast_prev_quarter), '#.#%') ||
    " | " || format_number(gap_directs_forecast, '$,###.#') as directs_fct_text
  , format_number(try_divide(gap_best_case, u.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_best_case, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_best_case, '$,###.#') as best_case_text
  , format_number(try_divide(worst_case_projection, u.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_worst_case, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_worst_case, '$,###.#') as worst_case_text
  , format_number(try_divide(submitted_weighted_projection, u.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_weighted_projection, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_weighted_projection, '$,###.#') as weighted_proj_text
  , format_number(try_divide(current_ds_forecast, u.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_ds_forecast, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_ds_forecast, '$,###.#') as ds_forecast_text
  , format_number(try_divide(t7d_flat_projection, u.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_t7d_flat_projection, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_t7d_flat_projection, '$,###.#') as t7d_proj_text
  , format_number(try_divide(t28d_flat_projection, u.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_t28d_flat_projection, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_t28d_flat_projection, '$,###.#') as t28d_proj_text

  from account_union_individuals as u
)

/* Forecast waterfall for current user only */
select * from quarterly_summary 
where view_name = 'Org Forecast'



 